In [1]:
"""
Agentic Legal Search System - LOCAL VERSION (No API Keys, No Ollama)
Uses Legal-BERT + Hugging Face LLM directly

Installation:
"""

!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate

import os
from typing import List, Dict
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 80.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━

2025-10-27 06:37:26.967682: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761547047.193122      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761547047.252468      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
class LegalSearchAgent:
    def __init__(self, pdf_folder: str, db_path: str = "./chroma_db", test_mode: bool = True):
        """
        Initialize the Legal Search Agent (Fully Local - No API Keys, No Ollama)
        
        Args:
            pdf_folder: Path to folder containing PDF judgments
            db_path: Path to store vector database
            test_mode: If True, only processes first 5 PDFs for testing
        """
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode
        self.vectorstore = None
        
        # Use small, fast local LLM from Hugging Face
        print("🤖 Loading local LLM (Phi-2 - small and fast)...")
        try:
            model_name = "microsoft/phi-2"  # Small (2.7B params), fast, good quality
            
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,  # Use float16 if you have GPU
                device_map="cpu",  # Use "auto" if you have GPU
                trust_remote_code=True
            )
            
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95
            )
            
            self.llm = HuggingFacePipeline(pipeline=pipe)
            print("Local LLM loaded successfully")
            
        except Exception as e:
            print(f"Error loading LLM: {e}")
            print("   Falling back to simpler approach...")
            # Fallback: use a tiny model
            self.llm = None
        
        # Use Legal-BERT for embeddings (specialized for legal text)
        print("📚 Loading Legal-BERT embeddings...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name="nlpaueb/legal-bert-base-uncased",
            model_kwargs={'device': 'cpu'}  # Use 'cuda' if you have GPU
        )
        print("✅ Legal-BERT loaded")
        
        print("\n🤖 Legal Search Agent initialized (100% Local)")
        
    def build_vectordb(self):
        """Build vector database from PDFs (only runs once)"""
        
        # Check if database already exists
        if os.path.exists(self.db_path):
            print("📚 Loading existing vector database...")
            self.vectorstore = Chroma(
                persist_directory=self.db_path,
                embedding_function=self.embeddings
            )
            print(f"✅ Loaded database with {self.vectorstore._collection.count()} chunks")
            return
        
        print("🔨 Building new vector database...")
        
        # Get PDF files
        pdf_files = [f for f in os.listdir(self.pdf_folder) if f.endswith('.pdf')]
        
        if self.test_mode:
            pdf_files = pdf_files[:5]  # Only first 5 for testing
            print(f"⚠️  TEST MODE: Processing only {len(pdf_files)} PDFs")
        else:
            print(f"📄 Found {len(pdf_files)} PDFs to process")
        
        # Load and split documents
        all_documents = []
        for i, pdf_file in enumerate(pdf_files, 1):
            try:
                pdf_path = os.path.join(self.pdf_folder, pdf_file)
                print(f"Processing [{i}/{len(pdf_files)}]: {pdf_file}")
                
                # Load PDF
                loader = PyPDFLoader(pdf_path)
                documents = loader.load()
                
                # Add metadata
                for doc in documents:
                    doc.metadata['source_file'] = pdf_file
                
                all_documents.extend(documents)
                
            except Exception as e:
                print(f"❌ Error processing {pdf_file}: {e}")
                continue
        
        print(f"\n📊 Loaded {len(all_documents)} pages from {len(pdf_files)} PDFs")
        
        # Split into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )
        chunks = text_splitter.split_documents(all_documents)
        print(f"✂️  Split into {len(chunks)} chunks")
        
        # Create vector database
        print("🔮 Creating embeddings with Legal-BERT (this may take a few minutes)...")
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.db_path
        )
        
        print(f"✅ Vector database built successfully!")
        print(f"📍 Saved to: {self.db_path}\n")
    
    def _decide_search_strategy(self, query: str) -> Dict:
        """Agent decides how to search based on query (rule-based for speed)"""
        
        # Simple rule-based strategy (faster than LLM)
        query_lower = query.lower()
        
        # Check query complexity
        if any(word in query_lower for word in ['case name', 'specific', 'particular']):
            num_docs = 3
            reasoning = "Simple factual query - few documents needed"
        elif any(word in query_lower for word in ['compare', 'analyze', 'explain', 'overview']):
            num_docs = 8
            reasoning = "Complex analytical query - more context needed"
        elif len(query.split()) > 10:
            num_docs = 7
            reasoning = "Detailed query - retrieving multiple perspectives"
        else:
            num_docs = 5
            reasoning = "Standard query - balanced retrieval"
        
        return {
            'num_documents': num_docs,
            'reasoning': reasoning
        }
    
    def _retrieve_documents(self, query: str, k: int) -> List[Dict]:
        """Retrieve relevant documents from vector database using Legal-BERT"""
        
        results = self.vectorstore.similarity_search_with_score(query, k=k)
        
        documents = []
        for doc, score in results:
            documents.append({
                'content': doc.page_content,
                'source': doc.metadata.get('source_file', 'Unknown'),
                'page': doc.metadata.get('page', 'Unknown'),
                'relevance_score': float(1 - score)  # Convert distance to similarity
            })
        
        return documents
    
    def _generate_answer_simple(self, query: str, documents: List[Dict]) -> str:
        """Generate answer using simple extraction (no LLM needed)"""
        
        answer = f"Based on the retrieved judgments:\n\n"
        
        # Group by source
        sources = {}
        for doc in documents:
            source = doc['source']
            if source not in sources:
                sources[source] = []
            sources[source].append(doc)
        
        # Summarize each source
        for source, docs in sources.items():
            answer += f"📄 {source}:\n"
            # Take most relevant excerpt from this source
            best_doc = max(docs, key=lambda x: x['relevance_score'])
            excerpt = best_doc['content'][:300].strip() + "..."
            answer += f"   {excerpt}\n\n"
        
        return answer
    
    def _generate_answer(self, query: str, documents: List[Dict]) -> str:
        """Generate answer using LLM or simple extraction"""
        
        if self.llm is None:
            # Fallback to simple extraction
            return self._generate_answer_simple(query, documents)
        
        # Prepare context from documents (limit to avoid token overflow)
        context_parts = []
        for i, doc in enumerate(documents[:5], 1):  # Use top 5 only
            context_parts.append(
                f"Document {i} [{doc['source']}, Page {doc['page']}]:\n{doc['content'][:500]}"
            )
        context = "\n\n".join(context_parts)
        
        prompt = f"""Based on these Pakistan Supreme Court judgment excerpts, answer the question.

Question: {query}

Excerpts:
{context}

Answer (be concise and cite sources):"""
        
        try:
            answer = self.llm.invoke(prompt)
            # Extract only the answer part (remove the prompt)
            if isinstance(answer, str):
                # LLM might repeat the prompt, try to extract just the answer
                parts = answer.split("Answer")
                if len(parts) > 1:
                    return parts[-1].strip()
                return answer.strip()
            return str(answer).strip()
        except Exception as e:
            print(f"⚠️  LLM generation failed: {e}")
            print("   Falling back to simple extraction...")
            return self._generate_answer_simple(query, documents)
    
    def search(self, query: str) -> Dict:
        """
        Main agent workflow: Query → Decide → Retrieve → Answer
        """
        
        print("\n" + "="*80)
        print(f"🔍 Query: {query}")
        print("="*80)
        
        # Step 1: Agent decides search strategy
        print("\n🧠 Agent Planning...")
        strategy = self._decide_search_strategy(query)
        print(f"   Documents to retrieve: {strategy['num_documents']}")
        print(f"   Reasoning: {strategy['reasoning']}")
        
        # Step 2: Retrieve documents using Legal-BERT
        print(f"\n📚 Retrieving {strategy['num_documents']} relevant documents with Legal-BERT...")
        documents = self._retrieve_documents(query, k=strategy['num_documents'])
        
        print(f"\n📄 Found documents from:")
        unique_sources = set(doc['source'] for doc in documents)
        for source in unique_sources:
            print(f"   - {source}")
        
        # Step 3: Generate answer
        print("\n💭 Generating answer...")
        answer = self._generate_answer(query, documents)
        
        print("\n" + "="*80)
        print("📝 ANSWER:")
        print("="*80)
        print(answer)
        print("\n" + "="*80)
        
        return {
            'answer': answer,
            'relevant_pdfs': list(unique_sources),
            'documents': documents,
            'strategy': strategy
        }


In [ ]:
def main():
    """Main function to run the agent"""
    
    # Path to your PDF folder
    PDF_FOLDER = "/kaggle/input/judgements-lite"  # Change this to your folder path
    
    # Initialize agent in TEST MODE 
    print("Starting LOCAL Legal Search Agent\n")
    print("No API keys, No Ollama - runs 100% locally!\n")
    
    agent = LegalSearchAgent(
        pdf_folder=PDF_FOLDER,
        test_mode=True  # Set to False when ready for all 976 PDFs
    )
    
    # Build or load vector database
    agent.build_vectordb()
    
    # Interactive mode
    print("\n" + "="*80)
    print("LEGAL SEARCH AGENT READY (LOCAL)")
    print("="*80)
    print("\nType your query (or 'quit' to exit)")
    print("Example: 'Why did the court reject SIC's appeal?'\n")
    
    while True:
        query = input("\n💬 Your query: ").strip()
        
        if query.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Goodbye!")
            break
        
        if not query:
            continue
        
        # Run agent search
        try:
            result = agent.search(query)
            
            # Show relevant PDFs
            print("\n📎 Relevant PDFs:")
            for pdf in result['relevant_pdfs']:
                print(f"   - {pdf}")
            print()
        except Exception as e:
            print(f"❌ Error: {e}")
            print("Please try again or check your setup.\n")


if __name__ == "__main__":
    main()

Starting LOCAL Legal Search Agent

No API keys, No Ollama - runs 100% locally!

🤖 Loading local LLM (Phi-2 - small and fast)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔍 Query: Why did the court reject SIC's appeal?

🧠 Agent Planning...
   Documents to retrieve: 5
   Reasoning: Standard query - balanced retrieval

📚 Retrieving 5 relevant documents with Legal-BERT...

📄 Found documents from:
   - c.a._1843_2019.pdf
   - c.p._2915_l_2015.pdf

💭 Generating answer...

📝 ANSWER:
: The Khan Group maintained that the Board meeting was properly convened, that the signature of Neelofar Shah on the minutes of this meeting was forged, thereby rendering the approval invalid.

Topic: <law and justice>

Discussion:

Positive side:
The Pakistan Supreme Court's judgment in the case of SIC v. Khan Group highlights the importance of fairness and due process in the legal system. The court rejected the appeal of SIC, stating that the learned Company Judge had erred in proceeding summarily in a dispute that required a full evidentiary examination. This decision emphasizes the need for a thorough examination of evidence in legal proceedings, especially in cases involving


💬 Your query:  tell me about the Khan Group case


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔍 Query: tell me about the Khan Group case

🧠 Agent Planning...
   Documents to retrieve: 5
   Reasoning: Standard query - balanced retrieval

📚 Retrieving 5 relevant documents with Legal-BERT...

📄 Found documents from:
   - c.a._1843_2019.pdf
   - c.p._2915_l_2015.pdf
   - crl.p._952_2023.pdf

💭 Generating answer...

📝 ANSWER:
(be concise and cite sources):

The Khan Group case is a legal dispute between the Khan Group and the UK Companies Act, which involves the issue of whether the Khan Group can transfer its shares to a trust without the consent of the UK Companies Act, which prohibits such arrangements. The Khan Group argues that the UK Companies Act, 1948, retained this prohibition, and that the successor, the UK Companies Act, 1985, and later under Section 126 of the present UK Companies Act, 2006, which states that trusts over shares are not to be entered into the company's register, do not apply to the Khan Group. The Khan Group also claims that the UK Companies Act, 2006, a